In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
file_path = "/content/market-crash.xlsx"
df = pd.read_excel(file_path, sheet_name='Worksheet')

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df['TIME'] = pd.to_datetime(df['TIME'])

In [ ]:
# Handle missing values (e.g., fill CPI with median)
df['CPI'].fillna(df['CPI'].median(), inplace=True)

In [ ]:
# Descriptive Statistics
print("Descriptive Statistics:")
print(df.describe())

In [ ]:
# Correlation heatmap (drop non-numeric columns)
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
# Prepare data for modeling
numerical_features = ['Unemployment', 'CPI', 'P/E', 'Open', 'High', 'Low', 'Close', 'Industrial Production', 'Treasury']
categorical_features = ['LOCATION']

In [ ]:
# Define preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [ ]:
# Split dataset into training and testing
X = df[numerical_features + categorical_features]
y = df['Crash']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Define the model pipeline with XGBoost
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42))
])

In [ ]:
# Train the model
model.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = model.predict(X_test)

In [ ]:
# Evaluation
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))